In [ ]:
# ============================================
# Cell 1: Install required libraries
# Run this cell once per new Colab session.
# ============================================

!pip install pandas numpy scikit-learn xgboost shap matplotlib joblib

In [ ]:
# ============================================
# Cell 2: Imports and global configuration
# This cell imports all the Python libraries we will use
# and sets some basic options (e.g. warnings, plotting).
# ============================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

from xgboost import XGBClassifier
import shap
import matplotlib.pyplot as plt
import joblib
import warnings

# Ignore minor warnings to keep the notebook output clean
warnings.filterwarnings("ignore")

# Make plots a bit sharper
plt.rcParams["figure.dpi"] = 120


In [ ]:
# ============================================
# Cell 3A: Load the dataset via manual upload
# (Use this if you will upload op_level_dataset_postop.csv each time)
# ============================================

from google.colab import files

print("Please upload op_level_dataset_postop.csv")
uploaded = files.upload()

# This assumes the uploaded file is called exactly this
df = pd.read_csv("op_level_dataset_postop.csv")

print("Raw shape:", df.shape)
df.head()


In [ ]:
df.isnull().sum()

In [ ]:
# ============================================
# Cell 4: Filter to relevant cases + derive BMI and operation duration
#
# This cell:
# 1. Restricts to elective cases (emop == 0).
# 2. Filters to orthopaedic departments (e.g., OS/OT).
# 3. Calculates BMI (if weight/height exist).
# 4. Calculates operation duration (opend_time - opstart_time).
# ============================================

print("Original dataset shape:", df.shape)

# 1. Keep only elective cases (assumption: emop == 0 is elective)
if "emop" in df.columns:
    df = df[df["emop"] == 0]
else:
    print("WARNING: Column 'emop' not found. Skipping elective filter.")

# 2. Restrict to orthopaedic departments (OS, OT etc.)
if "department" in df.columns:
    ortho_depts = ["OS", "OT"]
    df = df[df["department"].isin(ortho_depts)]
else:
    print("WARNING: Column 'department' not found. Skipping department filter.")

print("After elective + orthopaedic filter:", df.shape)

# 3. Compute BMI if height and weight exist
if set(["weight", "height"]).issubset(df.columns):
    df["bmi"] = np.where(
        df["height"].notna() & (df["height"] > 0),
        df["weight"] / (df["height"] / 100.0) ** 2,
        np.nan
    )
    print("BMI column created.")
else:
    print("WARNING: 'weight' or 'height' missing, cannot compute BMI.")

# 4. Compute operation duration if op times exist
if set(["opstart_time", "opend_time"]).issubset(df.columns):
    df["op_duration"] = df["opend_time"] - df["opstart_time"]
    # Replace non-sensical durations (<= 0) with NaN
    df.loc[df["op_duration"] <= 0, "op_duration"] = np.nan
    print("Operation duration column 'op_duration' created.")
else:
    print("WARNING: 'opstart_time' or 'opend_time' missing, cannot compute op_duration.")

df[["emop"] + [c for c in ["department", "bmi", "opstart_time", "opend_time", "op_duration"] if c in df.columns]].head()


In [ ]:
# ============================================
# Cell 5: Define the target variable (long_stay)
#
# We turn LOS (continuous) into a binary label:
#   long_stay = 1 if LOS > threshold, else 0.
# You can adjust the threshold to reflect your service.
# ============================================

# This column name must match your dataset
LOS_COL = "los_postop_days"

if LOS_COL not in df.columns:
    raise ValueError(f"Column '{LOS_COL}' not found. Check your dataset headers.")

# Define threshold for "long stay" in days (you can change this)
LONG_STAY_THRESHOLD = 7.0

df["long_stay"] = (df[LOS_COL] > LONG_STAY_THRESHOLD).astype(int)

print(df["long_stay"].value_counts())
print("Long-stay proportion:", df["long_stay"].mean())
df[[LOS_COL, "long_stay"]].head()


In [ ]:
# ============================================
# NEW CELL (after Cell 5): Subsample to 3,000 records
#
# 1. Keep the long_stay label.
# 2. Stratify the sampling so the proportion of long_stay vs non-long_stay
#    remains roughly the same as in the full dataset.
# 3. Use a fixed random_state for reproducibility.
# ============================================

MAX_RECORDS = 3000

print("Current number of records:", df.shape[0])

if df.shape[0] > MAX_RECORDS:
    # Use train_test_split trick to get a stratified sample of 3,000
    from sklearn.model_selection import train_test_split

    df_sample, _ = train_test_split(
        df,
        train_size=MAX_RECORDS,
        stratify=df["long_stay"],
        random_state=42
    )
    df = df_sample.copy()
    print("Downsampled to:", df.shape[0], "records (stratified by long_stay).")
else:
    print("Dataset has <= 3000 rows already, no downsampling applied.")

# Check the long_stay distribution in the sample
print(df["long_stay"].value_counts(normalize=True))


In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
# ============================================
# Cell 6: Select feature columns and split into X and y
#
# LEAKAGE FIX: use an explicit ALLOWLIST of predictors that are known
# before or during surgery. discharge_time, opstart_time and opend_time
# are deliberately excluded: LOS is derived from discharge_time, so the
# model was partly reading the answer. op_duration (derived in Cell 4)
# is kept. Race is excluded from the predictors and used only for the
# subgroup check.
# ============================================

ALLOWED_FEATURES = [
    "age", "sex", "bmi", "asa", "antype",
    "department", "icd10_pcs", "op_duration",
]

# Columns that must never be predictors (leakage / outcome / identifiers)
FORBIDDEN = {
    "discharge_time", "opstart_time", "opend_time",
    LOS_COL, "long_stay",
    "op_id", "subject_id", "hadm_id", "case_id",
}

feature_cols = [c for c in ALLOWED_FEATURES if c in df.columns]
missing = [c for c in ALLOWED_FEATURES if c not in df.columns]
if missing:
    print("WARNING: allowed features not found in data:", missing)

# Hard safety check: fail loudly if a leaky column slips in
leaks = FORBIDDEN.intersection(feature_cols)
assert not leaks, f"Leakage columns in feature set: {leaks}"

X = df[feature_cols].copy()
y = df["long_stay"].copy()

print("Number of features:", len(feature_cols))
print("Features:", feature_cols)
print("X shape:", X.shape, "| y shape:", y.shape)


In [ ]:
# ============================================
# New Cell C (optional): LOS distribution + long_stay proportions
#
# This gives:
# 1. Histogram of LOS (continuous).
# 2. Bar chart of long_stay vs non-long_stay counts.
# ============================================

plt.figure(figsize=(6, 4))
df[LOS_COL].hist(bins=30)
plt.xlabel("Postoperative LOS (days)")
plt.ylabel("Number of cases")
plt.title("Distribution of Postoperative LOS")
plt.tight_layout()
plt.show()

plt.figure(figsize=(4, 4))
df["long_stay"].value_counts().sort_index().plot(kind="bar")
plt.xticks([0, 1], ["No long stay", "Long stay"], rotation=0)
plt.ylabel("Number of cases")
plt.title("Class Distribution – Long stay vs No long stay")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================
# Cell 7: Define categorical & numeric features + preprocessing pipeline
#
# Preprocessing:
# - Numeric: median imputation + standard scaling.
# - Categorical: most frequent imputation + one-hot encoding.
# ============================================

# These are typical categorical candidates; we keep only ones that exist.
candidate_cats = ["sex", "race", "department", "antype", "icd10_pcs"]

categorical_features = [c for c in candidate_cats if c in X.columns]
numeric_features = [c for c in X.columns if c not in categorical_features]

print("Categorical features:", categorical_features)
print("Numeric features (first 20):", numeric_features[:20])

# Define transformations
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine into a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)


In [ ]:
# ============================================
# Cell 8: Train–test split
#
# We create a hold-out test set (30% of data), stratified by the long_stay label.
# This simulates unseen data for evaluation.
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    stratify=y,         # keep proportion of long_stay in train & test
    random_state=42
)

print("Train size:", X_train.shape, "| Test size:", X_test.shape)
print("Train long_stay proportion:", y_train.mean())
print("Test long_stay proportion:", y_test.mean())


In [ ]:
# ============================================
# Cell 9: Baseline model – Logistic Regression
#
# This gives you:
# - A simple, interpretable baseline.
# - Metrics to compare ML models against.
# ============================================

log_reg = LogisticRegression(max_iter=500)

log_reg_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", log_reg),
])

# Fit on training data
log_reg_pipeline.fit(X_train, y_train)

# Predictions on test set
y_pred_lr = log_reg_pipeline.predict(X_test)
y_proba_lr = log_reg_pipeline.predict_proba(X_test)[:, 1]

print("=== Logistic Regression Baseline ===")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("F1:", f1_score(y_test, y_pred_lr))
print("AUROC:", roc_auc_score(y_test, y_proba_lr))
print("AUPRC:", average_precision_score(y_test, y_proba_lr))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))


In [ ]:
# ============================================
# Cell 10: Main model – XGBoost classifier
#
# This is your primary ML model, aligned with your methodology:
# - Handles non-linearities and interactions.
# - Often performs very well on tabular clinical data.
# ============================================

xgb_clf = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)

xgb_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", xgb_clf),
])

xgb_pipeline.fit(X_train, y_train)

y_pred_xgb = xgb_pipeline.predict(X_test)
y_proba_xgb = xgb_pipeline.predict_proba(X_test)[:, 1]

print("=== XGBoost Model ===")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("F1:", f1_score(y_test, y_pred_xgb))
print("AUROC:", roc_auc_score(y_test, y_proba_xgb))
print("AUPRC:", average_precision_score(y_test, y_proba_xgb))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))


In [ ]:
# ============================================
# NEW CELL: 5-fold stratified cross-validation (training set only)
#
# A single 70/30 split on ~3,000 rows gives a noisy estimate.
# Report mean +/- SD across folds alongside the held-out test score.
# ============================================

from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"auroc": "roc_auc", "auprc": "average_precision"}

for name, pipe in [("Logistic Regression", log_reg_pipeline), ("XGBoost", xgb_pipeline)]:
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    print(f"{name}: "
          f"AUROC {res['test_auroc'].mean():.3f} +/- {res['test_auroc'].std():.3f} | "
          f"AUPRC {res['test_auprc'].mean():.3f} +/- {res['test_auprc'].std():.3f}")


In [ ]:
# ============================================
# New Cell A: ROC and Precision–Recall curves
#
# This cell visualises:
# 1. ROC curves for Logistic Regression vs XGBoost.
# 2. Precision–Recall curves for both models.
#
# These are excellent figures for Chapter 4.
# ============================================

from sklearn.metrics import roc_curve, precision_recall_curve

# --- ROC Curves ---
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_proba_xgb)

plt.figure(figsize=(6, 5))
plt.plot(fpr_lr, tpr_lr, label=f"Logistic Regression (AUROC = {roc_auc_score(y_test, y_proba_lr):.2f})")
plt.plot(fpr_xgb, tpr_xgb, label=f"XGBoost (AUROC = {roc_auc_score(y_test, y_proba_xgb):.2f})")
plt.plot([0, 1], [0, 1], "k--", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves – Long-stay Prediction")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# --- Precision–Recall Curves ---
prec_lr, rec_lr, _ = precision_recall_curve(y_test, y_proba_lr)
prec_xgb, rec_xgb, _ = precision_recall_curve(y_test, y_proba_xgb)

plt.figure(figsize=(6, 5))
plt.plot(rec_lr, prec_lr, label=f"Logistic Regression (AUPRC = {average_precision_score(y_test, y_proba_lr):.2f})")
plt.plot(rec_xgb, prec_xgb, label=f"XGBoost (AUPRC = {average_precision_score(y_test, y_proba_xgb):.2f})")
baseline = y_test.mean()
plt.hlines(baseline, 0, 1, colors="k", linestyles="dashed", label=f"Baseline (prevalence = {baseline:.2f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curves – Long-stay Prediction")
plt.legend(loc="upper right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================
# New Cell B: Confusion matrix heatmap for XGBoost
#
# This cell turns the confusion matrix into a nice
# annotated heatmap for your results section.
# ============================================

import seaborn as sns  # if not installed: !pip install seaborn

cm = confusion_matrix(y_test, y_pred_xgb)
cm_df = pd.DataFrame(
    cm,
    index=["Actual: No long stay", "Actual: Long stay"],
    columns=["Predicted: No long stay", "Predicted: Long stay"]
)

plt.figure(figsize=(5, 4))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix – XGBoost Model")
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================
# Cell 11: SHAP explainability – global and local
#
# This cell:
# - Extracts the fitted preprocessor and XGBoost model.
# - Computes SHAP values.
# - Creates a global summary plot (feature importance).
# - Creates a local explanation for one example patient.
# ============================================

# Extract fitted components from the pipeline
preprocessor_fitted = xgb_pipeline.named_steps["preprocess"]
model_fitted = xgb_pipeline.named_steps["model"]

# Transform training data into the numeric + one-hot encoded space
X_train_transformed = preprocessor_fitted.transform(X_train)

# Get feature names after preprocessing
cat_feature_names = []
if categorical_features:
    ohe = preprocessor_fitted.named_transformers_["cat"].named_steps["onehot"]
    cat_feature_names = ohe.get_feature_names_out(categorical_features)

feature_names = np.concatenate([numeric_features, cat_feature_names])

# Build SHAP explainer (TreeExplainer is optimised for tree-based models like XGBoost)
explainer = shap.TreeExplainer(model_fitted)
shap_values = explainer.shap_values(X_train_transformed)

# --- Global summary plot ---
plt.figure(figsize=(8, 6))
shap.summary_plot(
    shap_values,
    X_train_transformed,
    feature_names=feature_names,
    show=False
)
plt.title("Global SHAP Summary (XGBoost)")
plt.tight_layout()
plt.show()



In [ ]:
import shap
import numpy as np
import matplotlib.pyplot as plt

example_idx = 2
class_idx = 1  # for binary classification, try 1 (positive class)

# --- extract a single row of SHAP values + base value robustly ---
if isinstance(shap_values, shap.Explanation):
    # New SHAP API: shap_values is already an Explanation over the dataset
    exp = shap_values[example_idx]

else:
    # Old API: shap_values could be array or list
    if isinstance(shap_values, list):
        sv_row = shap_values[class_idx][example_idx]
        base = explainer.expected_value[class_idx] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value
    else:
        sv_row = shap_values[example_idx]
        base = explainer.expected_value  # scalar

    # If base is still array-like of length 1, unwrap it
    if isinstance(base, (np.ndarray, list)) and np.array(base).shape == (1,):
        base = float(base[0])

    # Build Explanation (attach X_test row if feature dims match)
    xrow = X_test.iloc[example_idx]
    if len(sv_row) == len(X_test.columns):
        exp = shap.Explanation(values=sv_row, base_values=base, data=xrow, feature_names=X_test.columns)
    else:
        # fallback if SHAP was computed on transformed features
        exp = shap.Explanation(values=sv_row, base_values=base)

# --- plot waterfall bigger ---
plt.figure(figsize=(12, 7))
shap.plots.waterfall(exp, max_display=12, show=False)
plt.title(f"Local SHAP Waterfall – Example index {example_idx}", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns  # if not installed: !pip install seaborn

def subgroup_metrics(X_test_df, y_true, y_proba, group_col, threshold=0.5):
    """
    Compute AUROC and F1-score for each subgroup of group_col.
    Skips very small subgroups to avoid noisy estimates.
    """
    if group_col not in X_test_df.columns:
        print(f"\n[Fairness] Column '{group_col}' not in X_test, skipping.")
        return

    y_pred = (y_proba >= threshold).astype(int)
    print(f"\n=== Fairness check by {group_col} ===")
    for group_value, group_idx in X_test_df[group_col].dropna().groupby(X_test_df[group_col]).groups.items():
        if len(group_idx) < 50:
            # Skip tiny groups; metrics would be unstable
            continue

        # Use .loc for label-based indexing on the pandas Series y_true (y_test)
        y_g = y_true.loc[group_idx]

        # Get positional indices for numpy arrays (y_proba, y_pred)
        # The index of y_true (y_test) and X_test_df are aligned.
        positional_indices = y_true.index.get_indexer(group_idx)

        proba_g = y_proba[positional_indices]
        pred_g = y_pred[positional_indices]

        try:
            auroc_g = roc_auc_score(y_g, proba_g)
        except ValueError:
            auroc_g = np.nan

        f1_g = f1_score(y_g, pred_g)

        print(f"{group_col} = {group_value} | n={len(group_idx)} | AUROC={auroc_g:.3f} | F1={f1_g:.3f}")


# Attach demographics from the original df
X_test_with_demo = X_test.copy()
for col in ["sex", "race"]:
    if col in df.columns:
        X_test_with_demo[col] = df.loc[X_test.index, col]

# Run fairness checks
subgroup_metrics(X_test_with_demo, y_test, y_proba_xgb, "sex")
subgroup_metrics(X_test_with_demo, y_test, y_proba_xgb, "race")

In [ ]:
# ============================================
# Cell 13: Save the trained XGBoost pipeline for later deployment
#
# This lets you:
# - Load the model in a separate Streamlit app.
# - Avoid retraining every time.
# ============================================

joblib.dump(xgb_pipeline, "los_xgb_pipeline.joblib")
joblib.dump(feature_cols, "feature_cols.joblib")

print("Saved model to 'los_xgb_pipeline.joblib' and feature list to 'feature_cols.joblib'")


In [ ]:
import gradio as gr
import pandas as pd
import numpy as np
import joblib
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import shap
import traceback

# ============================================================
# LOAD MODEL + PIPELINE
# ============================================================
def load_pipeline():
    pipeline = joblib.load("los_xgb_pipeline.joblib")
    feature_cols = joblib.load("feature_cols.joblib")
    return pipeline, feature_cols

pipeline, feature_cols = load_pipeline()
preprocessor = pipeline.named_steps["preprocess"]
model = pipeline.named_steps["model"]

numeric_features = preprocessor.transformers_[0][2]
categorical_features = preprocessor.transformers_[1][2]

# ============================================================
# HELPERS
# ============================================================
def to_timestamp_int(dt):
    return int(dt.strftime("%Y%m%d%H%M%S"))

def build_row(age, bmi, height_cm, asa, sex, race, anaesthetic, op_duration, icd10):
    height_m = height_cm / 100
    weight = bmi * (height_m ** 2)

    row = {
        "age": age,
        "weight": weight,
        "height": height_cm,
        "asa": asa,
        "emop": 0,
        "bmi": bmi,
        "op_duration": op_duration,
        "sex": sex,
        "race": race,
        "department": "OS",
        "antype": anaesthetic,
        "icd10_pcs": icd10,
    }

    return pd.DataFrame([{col: row.get(col, np.nan) for col in feature_cols}])


# ============================================================
# SHAP PLOT — FIXED FOR XGBOOST + COLAB + SPARSE MATRICES
# ============================================================
def generate_shap_plot(input_df):

    # Transform through preprocessor → then convert sparse to dense
    transformed_sparse = preprocessor.transform(input_df)

    transformed = (
        transformed_sparse.toarray()
        if hasattr(transformed_sparse, "toarray")
        else transformed_sparse
    )

    # Build SHAP explainer
    explainer = shap.TreeExplainer(model, feature_perturbation="interventional")
    shap_raw = explainer.shap_values(transformed)

    # Handle binary classifier + multi-output SHAP formats
    if isinstance(shap_raw, list):
        if len(shap_raw) > 1:  # positive class
            shap_values = np.array(shap_raw[1][0])
            base_val = float(np.array(explainer.expected_value)[1])
        else:
            shap_values = np.array(shap_raw[0][0])
            base_val = float(np.array(explainer.expected_value)[0])

    else:
        arr = np.array(shap_raw)

        if arr.ndim == 3:  # (samples, classes, features)
            if arr.shape[1] > 1:
                shap_values = arr[0, 1]
                base_val = float(np.array(explainer.expected_value)[1])
            else:
                shap_values = arr[0, 0]
                base_val = float(np.array(explainer.expected_value)[0])

        elif arr.ndim == 2:  # (samples, features)
            shap_values = arr[0]
            ev = np.array(explainer.expected_value).reshape(-1)
            base_val = float(ev[1] if len(ev) > 1 else ev[0])

        else:  # Already 1D
            shap_values = arr
            base_val = float(np.array(explainer.expected_value).reshape(-1)[0])

    shap_values = shap_values.reshape(-1)

    # Correct feature names
    ohe = preprocessor.named_transformers_["cat"].named_steps["onehot"]
    cat_names = ohe.get_feature_names_out(categorical_features)
    feature_names = np.concatenate([numeric_features, cat_names])

    # Ensure SHAP values align
    assert len(shap_values) == len(feature_names), \
        f"Mismatch: {len(shap_values)} values vs {len(feature_names)} names."

    # Build SHAP Explanation
    shap_exp = shap.Explanation(
        values=shap_values,
        base_values=base_val,
        data=transformed[0],
        feature_names=feature_names,
    )

    # Generate waterfall plot
    fig = plt.figure(figsize=(8, 6))
    shap.plots.waterfall(shap_exp, max_display=10, show=False)
    plt.tight_layout()
    return fig


# ============================================================
# MAIN MODEL PREDICTION
# ============================================================
def predict(age, height, bmi, asa, sex, race, anaesthetic, op_duration, icd10):
    try:
        # Build input row
        df = build_row(age, bmi, height, asa, sex, race, anaesthetic, op_duration, icd10)

        # Predict
        proba_arr = pipeline.predict_proba(df)
        prob = float(proba_arr[0][1])

        if prob < 0.20:
            risk_cat = "🟢 Low Risk"
        elif prob < 0.50:
            risk_cat = "🟠 Moderate Risk"
        else:
            risk_cat = "🔴 High Risk"

        shap_plot = generate_shap_plot(df)

        return f"{prob:.1%}", risk_cat, shap_plot

    except Exception as e:
        tb = traceback.format_exc()
        return f"Backend error:\n{e}\n\nTraceback:\n{tb}", "Error", None


# ============================================================
# GRADIO UI (FULL CLINICAL DASHBOARD)
# ============================================================
with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue")) as demo:

    gr.Markdown(
        """
        <h1 style='text-align:center; color:#0a2a66; font-weight:800;'>
            Arthroplasty LOS Risk Predictor
        </h1>
        <p style='text-align:center; color:#3f5468; font-size:17px;'>
            A clinical decision-support tool using Explainable AI (XGBoost + SHAP)
        </p>
        <br>
        """
    )

    with gr.Row():

        # ------------------- LEFT INPUT PANEL -------------------
        with gr.Column(scale=1):
            gr.Markdown("### 🧍‍♂️ Patient Profile")

            age = gr.Slider(18, 100, value=65, label="Age")
            height = gr.Slider(140, 210, value=170, label="Height (cm)")
            bmi = gr.Slider(15, 60, value=28, label="BMI")
            asa = gr.Dropdown([1, 2, 3, 4], value=2, label="ASA Class")

            gr.Markdown("### 🌍 Demographics")
            sex = gr.Dropdown(["M", "F"], value="M", label="Sex")
            race = gr.Dropdown(
                ["White", "Black", "Asian", "Other"],
                value="White",
                label="Race"
            )

            gr.Markdown("### 🏥 Surgical Details")
            anaesthetic = gr.Dropdown(
                ["General", "Spinal", "Combined"], value="General", label="Anaesthetic Type"
            )
            op_duration = gr.Slider(30, 300, value=90, label="Operative Duration (min)")
            icd10 = gr.Dropdown(
                ["0WJG0", "0DHS0", "0QB90"], value="0WJG0", label="ICD-10-PCS Code"
            )

            run_btn = gr.Button("🔍 Predict LOS Risk", variant="primary")


        # ------------------- RIGHT RESULTS PANEL -------------------
        with gr.Column(scale=1.2):

            gr.Markdown("### 📊 Results & Explainability")

            prob_out = gr.Textbox(label="Predicted Probability", lines=3)
            risk_out = gr.Textbox(label="Risk Category")
            shap_plot_out = gr.Plot(label="SHAP Waterfall Plot")

            run_btn.click(
                predict,
                inputs=[age, height, bmi, asa, sex, race, anaesthetic, op_duration, icd10],
                outputs=[prob_out, risk_out, shap_plot_out],
            )

    gr.Markdown(
        """
        <hr>
        <p style='text-align:center; color:#6c7a89; font-size:14px;'>
            Research prototype — not for clinical use
        </p>
        """
    )

demo.launch(share=True)
